# Red Five — compose your own report

Compute only the section you need. Select existing results for display, customize plots, and place them on your own axes. Synthetic data only; partial results never constitute an acceptance verdict.

In [ ]:
%matplotlib inline
from pathlib import Path

from IPython.display import display
from matplotlib.figure import Figure

from red_five.component_export import Component, export_components
from red_five.composition import PlotOptions, Selection
from red_five.rendering import verify_bundle
from red_five.sections import evaluate_section

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "examples/evaluation.json").is_file()
)
signals = (ROOT / "examples/signals.csv").read_bytes()
config = (ROOT / "examples/evaluation.json").read_bytes()
plan = (ROOT / "STATISTICAL_ANALYSIS_PLAN.md").read_bytes()
lock = (ROOT / "uv.lock").read_bytes()

## One section, no full report or weights required

The immutable result is reusable. Display and style changes below do not rerun evaluation. To change a numerical sample or date window, supply newly validated input/configuration to `evaluate_section`.

In [ ]:
standalone = evaluate_section("standalone", signals, config, plan, lock)
display(standalone)
print(standalone.section_id)

## Select one model and control the table

This is a display subset of already-computed rows, not a new correlation estimate. Omitted row counts remain visible. `table()` preserves exact values; `display(panel)` uses the chosen display precision.

In [ ]:
single = standalone.select(
    Selection(
        model_ids=("ES-model",),
        columns=("model_id", "instrument_id", "rank_ic", "eligible_observations"),
        precision=3,
    )
)
display(single)
display(single.table())
style = PlotOptions(
    figsize=(8, 4),
    colors=("#009E73",),
    markers=("D",),
    metrics=("rank_ic",),
    title="ES rank correlation",
    font_size=11,
)
display(single.figure(options=style))

## Compose sections on your own subplot axes

Coverage skips correlation computation. Existing figures/axes belong to you; the plotting API neither calls `show()` nor closes figures. All subplots keep their section metadata. Direct Matplotlib annotations are exploratory edits, not replayed by a standard component export.

In [ ]:
coverage = evaluate_section("coverage", signals, config, plan, lock)
fig = Figure(figsize=(13, 11), layout="constrained")
left = fig.add_subplot(211)
right = fig.add_subplot(212)
standalone.select().plot(ax=left, options=PlotOptions(title="Model comparison"))
coverage.select().plot(ax=right, options=PlotOptions(title="Coverage by model"))
display(fig)

## Economics independently

Only economics needs the externally supplied weights. It still validates alignment against the signal panel. Returns are interval accounting, not a portfolio NAV path. Sorting changes display order only.

In [ ]:
economics = evaluate_section(
    "economics",
    signals,
    config,
    plan,
    lock,
    weight_bytes=(ROOT / "examples/weights.csv").read_bytes(),
)
costs = economics.select(Selection(sort_by="net_return", descending=True))
display(costs.table())
display(costs.figure("costs", options=PlotOptions(figsize=(8, 4))))

## Export one table or a custom collection

Only selected components are rendered. Bundles include exact section JSON, display specifications, CSV, selected figures and a partial manifest. Use a new output path when changing a display specification; existing different evidence is never overwritten.

In [ ]:
output = ROOT / "build" / f"composition-{standalone.section_id}"
single.export(output / "one-table")
export_components(
    [
        Component(single, "correlations", style),
        Component(costs, "costs", PlotOptions(title="Supplied costs")),
    ],
    output / "selected-report",
)
assert verify_bundle(output / "one-table")["scope"] == "partial"
assert verify_bundle(output / "selected-report")["scope"] == "partial"
print(output)

## Use an existing full report instead

Call `load_report(path).section('standalone')` to extract the same section API without reevaluating. Sections extracted from reports preserve their source-report lineage; independent sections bind their input, plan, configuration, code and lock identities directly. Different lineage may give different section IDs even when table values agree.

Optional widgets, arbitrary layout serialization, and automatic capture of manual Matplotlib edits are not implemented. Static APIs need no widget service or dashboard.